# Forelesning 2: Datahåndtering med Pandas

<div class="bilde-midt" style="width:32%">

![Panda](bilder/panda.jpg)
<p class="kreditering">Foto: [kilde og lisens]</p>

</div>

Navnet Pandas kommer faktisk fra *panel data*, ikke fra dyret.

I denne forelesningen skal vi:

- lage enkle, gode figurer av kjemiske måledata med Matplotlib
- lese datafiler med Pandas og undersøke hva vi faktisk har lest
- håndtere manglende verdier uten å ødelegge rådataene
- filtrere, sortere og gruppere data
- lage nye kolonner, for eksempel blankkorrigert absorbans
- gå fra dataramme til figur

<div class="bilde-hoyre" style="width:34%">

![UV-Vis-spektrofotometer](bilder/spektrofotometer.jpg)
<p class="kreditering">Foto: [kilde og lisens]</p>

</div>

Gjennomgangseksempelet er et UV-Vis-forsøk med blankprøver, kalibreringsstandarder og tre replikater av en ukjent prøve. Seinere i emnet skal vi bruke standardene til å bestemme konsentrasjonen i den ukjente prøven. I dag gjør vi dataene klare for det.

Menti: [lenke]

In [ ]:
import matplotlib.pyplot as plt

# Større figurer og skrift, slik at plottene er lesbare på lysbildene
plt.rcParams.update({"font.size": 14, "figure.figsize": (8, 4.5)})

## Del 1: Gode kjemiske figurer

<div class="viktig">

Før vi lager en figur, bør vi spørre:

- Hva representerer hvert datapunkt?
- Hvilken variabel har vi kontrollert, og hvilken har vi målt?
- Følger punktene en naturlig rekkefølge, for eksempel i tid?

</div>

Damptrykket til vann er målt ved fem temperaturer. Temperaturen er valgt, og damptrykket er målt.

| Temperatur (°C) | 16 | 18 | 20 | 22 | 24 |
|---|---|---|---|---|---|
| Damptrykk (kPa) | 1,817 | 2,063 | 2,339 | 2,644 | 2,984 |

In [ ]:
temperature_C = [16, 18, 20, 22, 24]
vapor_pressure_kPa = [1.817, 2.063, 2.339, 2.644, 2.984]

plt.scatter(temperature_C, vapor_pressure_kPa, label="Measurements")
plt.xlabel("Temperature (°C)")
plt.ylabel("Vapor pressure (kPa)")
plt.title("Vapor pressure of water")
plt.legend()
plt.show()

### Skal punktene bindes sammen?

<div class="viktig">

Bruk **punktplott** (`scatter`) for enkeltmålinger og sammenhenger mellom to størrelser. Bruk **linjer** (`plot`) når rekkefølgen er viktig, for eksempel i en tidsserie, eller når linja er en modell. En linje mellom målepunkter betyr ikke at vi har målt verdiene mellom punktene.

</div>

Absorbansen er målt gjennom en reaksjon ved to temperaturer.

In [ ]:
time_s = [0, 20, 40, 60, 80, 100]
absorbance_298_K = [0.812, 0.552, 0.378, 0.260, 0.181, 0.128]
absorbance_308_K = [0.809, 0.476, 0.284, 0.172, 0.108, 0.071]

plt.plot(time_s, absorbance_298_K, marker="o", label="298 K")
plt.plot(time_s, absorbance_308_K, marker="s", label="308 K")
plt.xlabel("Time (s)")
plt.ylabel("Absorbance")
plt.legend()
plt.show()

<div class="diskuter">

**Diskuter:** Hvilken blanding reagerer raskest? Er figuren alene nok til å konkludere med at temperaturen er årsaken?

</div>

<div class="oppgave">

**Prøv selv:** Programmet nedenfor lager en figur som er vanskelig å tolke. Anta at `x` er tid i timer og `y` er pH. Finn minst fire forbedringer.

Sjekkliste: navn og enhet på begge akser, riktig variabel på hver akse, farger som synes, markører på målepunktene og en tittel som sier noe faglig.

</div>

In [ ]:
time_h = [0, 1, 2, 3, 4]
pH = [7.02, 6.81, 6.60, 6.42, 6.25]

plt.plot(time_h, pH, marker="o", color="#176b87")
plt.xlabel("Time (h)")
plt.ylabel("pH")
plt.title("pH of the solution over time")
plt.show()

## Del 2: Datafiler

Store datasett skriver vi ikke inn i lister for hånd. De ligger i datafiler, ofte som CSV (*comma-separated values*). Her er starten av fila `uvvis_raw.csv`, som råtekst og slik Pandas leser den:

<div class="bilde-midt" style="width:95%">

![CSV som råtekst og som dataramme](bilder/csv_og_dataramme.png)

</div>

<div class="viktig">

**Ryddig dataformat:** Hver rad er én observasjon, hver kolonne er én variabel, og hver celle inneholder én verdi. Enheten står i kolonnenavnet (`concentration_uM`), mens cellene bare inneholder tall.

</div>

<div class="diskuter">

**Diskuter:** Hvorfor er det dårlig å skrive `"2 µmol/L"` i cellene i stedet for `2`?

</div>

## Del 3: Lese og undersøke data

Pandas leser tabelldata og gir oss en *dataramme* (`DataFrame`).

<div class="viktig">

En **dataramme** er en tabell med navngitte kolonner og rader. I motsetning til en tabell i et regneark kan vi behandle hele kolonner med kode.

</div>

Notatboka ligger i mappa `forelesninger`, mens datafilene ligger i `datahandtering_visualisering/data`. `..` betyr «gå én mappe opp».

In [ ]:
import pandas as pd

raw_data = pd.read_csv("../datahandtering_visualisering/data/uvvis_raw.csv")
raw_data

### Hva har vi egentlig lest?
Før vi analyserer noe, sjekker vi hva Pandas har lest.

| Kode | Gir |
|---|---|
| `raw_data.head()` / `raw_data.tail()` | de fem første / siste radene |
| `raw_data.shape` | antall rader og kolonner |
| `raw_data.columns` | kolonnenavnene |
| `raw_data.dtypes` | datatypen til hver kolonne |
| `raw_data.info()` | alt dette samlet, pluss antall verdier som ikke mangler |

In [ ]:
print(raw_data.shape)
print(raw_data.columns)
raw_data.info()

<div class="diskuter">

**Diskuter:** Konsentrasjonen mangler for den ukjente prøven. Er det en feil? Hvor finner du den andre manglende verdien, og er den en feil?

</div>

### Kolonner
En kolonne hentes med navnet i hakeparenteser. Resultatet er en *serie*, og den har metoder for vanlige beregninger.

In [ ]:
absorbance = raw_data["absorbance"]
print(absorbance.mean())
print(absorbance.max())
raw_data["sample_type"].unique()

<div class="oppgave">

**Prøv selv:** Fila `example_semicolon.csv` er lagret slik norske regneark ofte gjør det: med semikolon som skilletegn og komma som desimaltegn. Kjør koden. Hva har gått galt? Rett den opp med argumentene `sep` og `decimal` i `read_csv`, slik at konsentrasjonen blir tall.

</div>

In [ ]:
semicolon_data = pd.read_csv("../datahandtering_visualisering/data/example_semicolon.csv", sep=";", decimal=",")
print(semicolon_data)
print(semicolon_data.dtypes)

## Del 4: Rydde data

<div class="viktig">

**Bevar rådataene.** Vi endrer aldri `raw_data`. Alle ryddevalg gjøres på en kopi, og koden dokumenterer hva vi har gjort:

1. `raw_data`: dataene slik de ble lest
2. `clean_data = raw_data.copy()`: kopien vi rydder i
3. Analysen gjøres på `clean_data`

</div>

### Manglende verdier
Pandas markerer manglende verdier som `NaN` (*not a number*).

In [ ]:
print(raw_data.isna().sum())

raw_data[raw_data["absorbance"].isna()]

<div class="viktig">

`NaN` og `0` er ikke det samme. Null kan være et gyldig måleresultat, mens `NaN` betyr at verdien ikke finnes. Det finnes ingen automatisk riktig måte å håndtere en manglende verdi på. Valget må begrunnes faglig.

</div>

I instrumentloggen står det at målingen `std_6_3` mislyktes fordi kyvetten sto feil. Det er en god grunn til å utelate akkurat denne raden.

<div class="oppgave">

**Prøv selv:** Tipp først: Hva skjer hvis vi bruker `dropna()` *uten* argumenter på hele datasettet? Kjør koden og se om du hadde rett. Bruk deretter `subset` slik at bare raden med manglende absorbans fjernes.

</div>

In [ ]:
print(raw_data.dropna())

clean_data = raw_data.copy()
clean_data = clean_data.dropna(subset=["absorbance"])
print(clean_data.shape)

## Del 5: Filtrere, sortere og gruppere

Et logisk vilkår på en kolonne gir en serie med `True` og `False`. Når vi bruker serien inne i hakeparentesene, beholdes bare radene der svaret er `True`.

In [ ]:
print(clean_data["sample_type"] == "standard")

standards = clean_data[clean_data["sample_type"] == "standard"]
standards

Flere vilkår kombineres med `&` (og), `|` (eller) og `~` (ikke). **Hvert vilkår må stå i parentes.**

```python
selected = clean_data[
    (clean_data["sample_type"] == "standard")
    & (clean_data["concentration_uM"] >= 4)
]
```

<div class="oppgave">

**Prøv selv:** Velg ut

1. alle blankprøvene
2. alle standardene med konsentrasjon mindre enn eller lik 6 µmol/L
3. alle rader som ikke tilhører den ukjente prøven

</div>

In [ ]:
blanks = clean_data[clean_data["sample_type"] == "blank"]

low_standards = clean_data[
    (clean_data["sample_type"] == "standard")
    & (clean_data["concentration_uM"] <= 6)
]

known_samples = clean_data[clean_data["sample_type"] != "unknown"]

print(blanks)
print(low_standards)
print(known_samples)

### Sortere
Sortering endrer ikke dataene, men gjør det lettere å oppdage uventede verdier.

In [ ]:
clean_data.sort_values(by=["sample_type", "concentration_uM", "replicate"])

### Gruppere og oppsummere
Standardene er målt i tre replikater. `groupby` deler radene i grupper med samme konsentrasjon, og `agg` bestemmer hva som skal beregnes for hver gruppe.

In [ ]:
summary = standards.groupby("concentration_uM")["absorbance"].agg(["count", "mean", "std"])
summary

<div class="oppgave">

**Prøv selv:** Lag en oppsummering som også inneholder `min` og `max`.

</div>

In [ ]:
summary = standards.groupby("concentration_uM")["absorbance"].agg(["count", "mean", "std", "min", "max"])
summary

<div class="diskuter">

**Diskuter:** Hvorfor kan vi ikke beregne et standardavvik fra én måling? Er det grunnlag for å si at spredningen øker med konsentrasjonen?

</div>

## Del 6: Nye kolonner

Vi korrigerer absorbansen for bidraget fra blankprøvene. Resultatet lagres i en **ny kolonne**, slik at den opprinnelige målingen beholdes.

`loc[radvilkår, kolonnenavn]` velger bestemte rader og én kolonne samtidig.

In [ ]:
blank_mean = clean_data.loc[clean_data["sample_type"] == "blank", "absorbance"].mean()
print(blank_mean)

clean_data["blank_corrected_absorbance"] = clean_data["absorbance"] - blank_mean
clean_data

<div class="viktig">

Blankkorreksjon er et **modellvalg**, ikke bare en Pandas-operasjon. Vi antar at blankbidraget legges til signalet fra prøven, og at blankprøvene er representative for alle målingene.

</div>

<div class="oppgave">

**Prøv selv:** Kontroller beregningen. Gjennomsnittet av den blankkorrigerte absorbansen for blankprøvene skal være omtrent null.

</div>

In [ ]:
corrected_blanks = clean_data.loc[clean_data["sample_type"] == "blank", "blank_corrected_absorbance"]
print(corrected_blanks.mean())

## Del 7: Fra dataramme til figur

Kolonner i en dataramme kan sendes rett til Matplotlib.

In [ ]:
standards = clean_data[clean_data["sample_type"] == "standard"]

plt.scatter(standards["concentration_uM"], standards["blank_corrected_absorbance"])
plt.xlabel("Concentration (µmol/L)")
plt.ylabel("Blank-corrected absorbance")
plt.title("UV-Vis calibration standards")
plt.show()

<div class="oppgave">

**Prøv selv:** Plott gjennomsnittet for hver konsentrasjon med standardavviket som feilstolper. Bruk `summary` fra gruppene ovenfor og `plt.errorbar(x, y, yerr=..., fmt="o", capsize=4)`. Konsentrasjonene ligger i `summary.index`.

</div>

In [ ]:
plt.errorbar(summary.index, summary["mean"], yerr=summary["std"], fmt="o", capsize=4)
plt.xlabel("Concentration (µmol/L)")
plt.ylabel("Absorbance")
plt.show()

<div class="diskuter">

**Diskuter:** Hvilken informasjon mister vi når vi plotter gjennomsnitt i stedet for alle replikatene? Hvorfor er det ikke nok å skrive «error bars» i figurteksten?

</div>

## Del 8: Rydde tekst og finne duplikater (hvis tid)

Virkelige datafiler har ofte mellomrom, store og små bokstaver om hverandre og rader som er lagret to ganger. Vi lager et lite, rotete datasett fra en dictionary for å se hvordan vi kan rydde.

In [ ]:
messy_data = pd.DataFrame({
    " Sample ID": ["std_2_1", "std_2_2", "std_2_2", "unknown_1"],
    "Sample Type ": ["Standard", "standard ", "standard ", "UNKNOWN"],
    "Absorbance": [0.171, 0.169, 0.169, 0.603],
})
messy_data

In [ ]:
tidy_data = messy_data.copy()
tidy_data.columns = tidy_data.columns.str.strip().str.lower().str.replace(" ", "_")
tidy_data["sample_type"] = tidy_data["sample_type"].str.strip().str.lower()

print(tidy_data["sample_type"].unique())
tidy_data[tidy_data.duplicated(subset=["sample_id"], keep=False)]

<div class="diskuter">

**Diskuter:** To rader har samme absorbans. Når er det en duplikat som bør fjernes, og når er det to ekte replikater?

</div>

## En reproduserbar arbeidsflyt

<div class="viktig">

1. Les rådataene og ikke endre dem
2. Undersøk: `info()`, `isna().sum()`
3. Lag en arbeidskopi med `copy()`
4. Gjør begrunnede ryddevalg, for eksempel `dropna(subset=...)`
5. Lag nye kolonner i stedet for å overskrive målinger
6. Lagre resultatet i en **ny** fil

</div>

Koden dokumenterer *hva* vi har gjort. Vi må i tillegg kunne forklare *hvorfor* valgene er rimelige.

In [ ]:
clean_data.to_csv("uvvis_processed.csv", index=False)

<div class="diskuter">

**Menti:** Hvilket av ryddevalgene i dag synes du var vanskeligst å begrunne faglig?

</div>

__Oppgaver__:

<div class="bilde-hoyre" style="width:26%">

![Flammeprøver](bilder/flammeprover.jpg)
<p class="kreditering">Foto: [kilde og lisens]</p>

</div>

1. Les `reaction_kinetics.csv`, undersøk datatypene og sorter etter tid. Lag en ny kolonne `relative_absorbance` der hver absorbans deles på startverdien. Plott den relative absorbansen mot tid og beskriv utviklingen med kjemiske ord.
2. Fila `flame_emission.csv` inneholder emisjonsintensitet ved 589, 671 og 766 nm for prøver med Na⁺, Li⁺ og K⁺. Undersøk datasettet, og beregn gjennomsnittlig emisjon for hvert ion ved hver bølgelengde med `groupby`. Hvilket ion emitterer sterkest ved hver bølgelengde? Stemmer det med flammefargene du kjenner?
3. Plott alle kalibreringsstandardene med ulik markør for hver replikatserie. Blir figuren mer eller mindre informativ?
4. Lag en figur der en misvisende y-akse får to nesten like måleserier til å se svært forskjellige ut. Lag deretter en redelig versjon, og forklar valgene dine.
5. Skriv et fullstendig program som leser `uvvis_raw.csv`, undersøker manglende verdier, lager en arbeidskopi, fjerner den dokumenterte mislykkede målingen, blankkorrigerer og lagrer resultatet i en ny fil. Programmet skal aldri endre rådatafila.

Alle datafilene ligger i `../datahandtering_visualisering/data/`.